# 05 — Feature Engineering

Biến dữ liệu thô thành các feature dùng cho Classification và Regression. Các feature lịch sử được tính theo thời gian để tránh nhìn thấy tương lai.

In [ ]:
import os
import pandas as pd
import numpy as np

file_path = '../data/raw/parking_data_raw.csv'

df = pd.read_csv(file_path)
df['entry_time'] = pd.to_datetime(df['entry_time'])
df['exit_time'] = pd.to_datetime(df['exit_time'])

print("Kích thước dữ liệu thô:", df.shape)

In [ ]:
# 1. Feature thời gian
df['entry_hour'] = df['entry_time'].dt.hour
df['entry_minute'] = df['entry_time'].dt.minute
df['day_of_week_num'] = df['entry_time'].dt.weekday
df['is_weekend'] = df['day_of_week_num'].isin([5, 6]).astype(int)
df['is_morning'] = (df['entry_hour'] < 12).astype(int)

display(
    df[
        ['entry_time', 'entry_hour', 'entry_minute',
         'day_of_week_num', 'is_weekend', 'is_morning']
    ].head()
)

In [ ]:
# 2. Feature lịch sử theo từng sinh viên
# Phải sort trước để mọi feature chỉ dùng thông tin đã xảy ra trước đó.
df = df.sort_values(
    by=['student_id', 'entry_time']
).reset_index(drop=True)

# Trung bình thời gian đỗ của tối đa 5 lần trước.
df['rolling_avg_duration'] = (
    df.groupby('student_id')['duration_minutes']
      .transform(
          lambda x: x.shift(1).rolling(window=5, min_periods=1).mean()
      )
)

global_duration_mean = df['duration_minutes'].mean()

df['rolling_avg_duration'] = (
    df['rolling_avg_duration'].fillna(global_duration_mean)
)

# Số lần từng đỗ qua đêm trước lần gửi hiện tại.
df['is_overnight_flag'] = (
    df['parking_behavior'] == 'Overnight'
).astype(int)

df['historical_overnight_count'] = (
    df.groupby('student_id')['is_overnight_flag']
      .transform(
          lambda x: x.shift(1).fillna(0).cumsum()
      )
)

df = df.drop(columns=['is_overnight_flag'])

display(
    df[
        ['student_id', 'entry_time', 'duration_minutes',
         'rolling_avg_duration', 'historical_overnight_count']
    ].head(10)
)

In [ ]:
# 3. One-hot encoding cho các biến phân loại
df = pd.get_dummies(
    df,
    columns=['vehicle_type', 'usual_zone'],
    drop_first=True
)

dummy_cols = [
    col for col in df.columns
    if col.startswith('vehicle_type_') or col.startswith('usual_zone_')
]

df[dummy_cols] = df[dummy_cols].astype(int)

print("Các cột sau Feature Engineering:")
print(df.columns.tolist())

display(df.head())

In [ ]:
# 4. Tạo dataset riêng cho từng bài toán
os.makedirs('../data/processed', exist_ok=True)

drop_cols_clf = [
    'student_id',
    'entry_time',
    'exit_time',
    'day_of_week',
    'duration_minutes'
]

df_classification = df.drop(columns=drop_cols_clf)

drop_cols_reg = [
    'student_id',
    'entry_time',
    'exit_time',
    'day_of_week',
    'parking_behavior'
]

df_regression = df.drop(columns=drop_cols_reg)

classification_path = '../data/processed/classification_data.csv'
regression_path = '../data/processed/regression_data.csv'

df_classification.to_csv(classification_path, index=False)
df_regression.to_csv(regression_path, index=False)

print("Đã lưu:")
print("-", classification_path)
print("-", regression_path)

print("
Classification:", df_classification.shape)
print("Regression:", df_regression.shape)

In [ ]:
# 5. Kiểm tra cuối
print("Classification columns:")
print(df_classification.columns.tolist())

print("
Regression columns:")
print(df_regression.columns.tolist())

print("
Target Classification: parking_behavior")
print(df_classification['parking_behavior'].value_counts())

print("
Target Regression: duration_minutes")
print(df_regression['duration_minutes'].describe().round(2))